In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq

In [4]:
llm=ChatGroq(model="deepseek-r1-distill-llama-70b")

In [6]:
import os
print(os.getenv("GROQ_API_KEY")[:8] if os.getenv("GROQ_API_KEY") else "NOT FOUND")

your_api


In [12]:
llm.invoke

<bound method BaseChatModel.invoke of ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'DeepSeek R1 Distill Llama 70B', 'status': 'deprecated', 'release_date': '2025-01-20', 'last_updated': '2025-01-20', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000234ACCF6FC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000234AF16C5F0>, model_name='deepseek-r1-distill-llama-70b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)>

In [13]:
from langchain.tools import tool

In [15]:
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two integers.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The product of a and b.
    """
    return a * b

In [16]:
multiply

StructuredTool(name='multiply', description='Multiply two integers.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000234AFFB0900>)

In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import ClassVar, Type

In [18]:
class WeatherInput(BaseModel):
    city: str

In [19]:
def get_weather(city: str, units: str = "metric") -> str:
    """
    Get the weather for a given city.

    Args:
        city (str): The name of the city.
        units (str): 'metric' or 'imperial'.

    Returns:
        str: A string describing the weather in the city.
    """
    return f"The weather in {city} is sunny. ({units})"

In [20]:
weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetches real-time weather data for a city",
    args_schema=WeatherInput,  
)

In [22]:
class WeatherInput(BaseModel):
    city: str = Field(..., description="City name")
    units: str = Field("metric", description="metric or imperial")

class GetWeatherTool(StructuredTool):
    name: ClassVar[str] = "get_weather"           
    description: ClassVar[str] = (
        "Fetches weather data for a city"
    )
    args_schema: ClassVar[Type[BaseModel]] = WeatherInput

    def _run(self, city: str, units: str = "metric") -> str:
        return get_weather(city, units)